In [5]:
!unzip /content/pretrain.zip

Archive:  /content/pretrain.zip
   creating: pretrain/
  inflating: pretrain/test_encyclopedia.json  
  inflating: pretrain/train_encyclopedia.json  


In [2]:
!pip install datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 480.6/480.6 kB 20.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.3/179.3 kB 17.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.1/194.1 kB 12.1 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2024.10.0
    Uninstalling fsspec-2024.10.0:
      Successfully uninstalled fsspec-2024.10.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2024.10.0 requires fsspec==2024.10.0, but you have fsspec 2024.9.0 which is incompatible.


In [4]:
import torch
import os

os.environ['HF_ENDPOINT'] = 'https://hf-mirror.com'
device = 'cuda' if torch.cuda.is_available() else 'cpu'

from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained('gpt2')
tokenizer.pad_token = tokenizer.eos_token

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

In [6]:
from datasets import load_dataset
from transformers import default_data_collator

dataset = load_dataset('json', data_files='/content/pretrain/test_encyclopedia.json',split = 'train')
print(dataset)

def f(data):
    data = [i['text'] for i in data]

    data = tokenizer(data,
                     padding=True,
                     truncation=True,
                     max_length=50,
                     return_tensors='pt').to(device)

    data['labels'] = data['input_ids'].clone()
    select = data['labels'] == tokenizer.pad_token_id
    data['labels'][select] = -100

    return data


loader = torch.utils.data.DataLoader(dataset,
                                     batch_size=4,
                                     shuffle=True,
                                     drop_last=True,
                                     collate_fn=f)

len(loader), next(iter(loader))

Generating train split: 0 examples [00:00, ? examples/s]

Dataset({
    features: ['text'],
    num_rows: 500
})


(125,
 {'input_ids': tensor([[  163,   118,    95, 19021,   106,   163,   247,    96, 27670,   248,
          27670,   254,   162,   253,   241, 28938,   245,   171,   120,   253,
            163,   118,    95, 19021,   106,   163,   247,    96, 42468, 18796,
            109,   162,    96,   240,   163,   232,   114, 30266,   228,   164,
            237,   234,   161,   109,   252, 21410, 36181,   106,   163,   119],
         [33566,   112,   164,   224,   254,   164,   227,   118,   163,   246,
             97,   163,   245,   229,   163,   232,   114,   164,   115,   253,
            163,   245,   242,   163,   244,   106, 31660, 43718,   115, 28938,
            245,   171,   120,   253, 33566,   112,   164,   224,   254,   164,
            227,   118,   163,   246,    97,   161,   240,   234,   163,   245],
         [  164,   226,   239,   164,   228,   101, 49035,   118,   161,   240,
            234,   164,   226,   239,   164,   226,   232,   164,   228,   250,
            164,  

In [7]:
from transformers import AutoModelForCausalLM

model_actor = AutoModelForCausalLM.from_pretrained('gpt2').to(
    device)

model_actor.config


GPT2Config {
  "_attn_implementation_autoset": true,
  "_name_or_path": "gpt2",
  "activation_function": "gelu_new",
  "architectures": [
    "GPT2LMHeadModel"
  ],
  "attn_pdrop": 0.1,
  "bos_token_id": 50256,
  "embd_pdrop": 0.1,
  "eos_token_id": 50256,
  "initializer_range": 0.02,
  "layer_norm_epsilon": 1e-05,
  "model_type": "gpt2",
  "n_ctx": 1024,
  "n_embd": 768,
  "n_head": 12,
  "n_inner": null,
  "n_layer": 12,
  "n_positions": 1024,
  "reorder_and_upcast_attn": false,
  "resid_pdrop": 0.1,
  "scale_attn_by_inverse_layer_idx": false,
  "scale_attn_weights": true,
  "summary_activation": null,
  "summary_first_dropout": 0.1,
  "summary_proj_to_labels": true,
  "summary_type": "cls_index",
  "summary_use_proj": true,
  "task_specific_params": {
    "text-generation": {
      "do_sample": true,
      "max_length": 50
    }
  },
  "transformers_version": "4.47.1",
  "use_cache": true,
  "vocab_size": 50257
}

In [12]:
optimizer = torch.optim.Adam(model_actor.parameters(), lr=1e-5)

for i, data in enumerate(loader):
    out = model_actor(**data)
    out.loss.backward()
    optimizer.step()
    optimizer.zero_grad()

    if i % 10 == 0:
        print(i, len(loader), out.loss.item())

        prompt = data['input_ids'][0]
        chosen = prompt[50:]
        prompt = prompt[:50]

        gen = model_actor.generate(prompt.unsqueeze(0),
                                   max_length=300,
                                   pad_token_id=tokenizer.pad_token_id,
                                   eos_token_id=tokenizer.eos_token_id)[0, 50:]

        print('prompt=', tokenizer.decode(prompt))
        print('chosen=', tokenizer.decode(chosen))
        print('gen=', tokenizer.decode(gen))

model_actor.save_pretrained('model/actor')

0 125 2.0853676795959473
prompt= 女性治疗尿频尿痛尿血的费用大概多少钱？考虑是�
chosen= 
gen= �意的治疗尿频尿血，少钱，少钱，少钱，少钱，少钱，少钱，少钱，少钱，少钱，少钱，少钱，少钱，少钱，少钱，少钱，少钱，少钱，少钱，少钱，少钱，少钱，少钱，少钱，少钱，少钱，少钱，少钱，少钱，少�
10 125 1.8672242164611816
prompt= 飞龙掌血是什么?？飞龙掌血，为芸香科飞龙掌血
chosen= 
gen= ，为芸香科飞龙掌血，为芸香科飞龙掌血，为芸香科飞龙掌血，为芸香科飞龙掌血，为芸香科飞龙掌血，为芸香科飞龙掌血，为芸香科飞龙掌血，为芸香科飞龙掌血，为芸香科飞龙掌血，为芸香科飞龙掌血，为�5香科飞龙�
20 125 1.8581714630126953
prompt= 请问做锥切手术过程疼吗？宫颈锥切术是治疗宫�
chosen= 
gen= ��锥切的治疗，治疗宫颈锥切的治疗，治疗宫颈锥切的治疗，治疗宫颈锥切的治疗，治疗宫颈锥切的治疗，治疗宫颈锥切的治疗，治疗宫颈锥切的治疗，治疗宫颈锥切的治疗，治疗宫颈锥切的治疗，治疗宫颈锥切的治疗�
30 125 1.9718115329742432
prompt= 女性肺癌的症状和前兆？在很多人的心中，肺癌一定是
chosen= 
gen= 一种常见的症状和前，肺癌一定是一种常见的症状和前，肺癌一定是一种常见的症状和前，肺癌一定是一种常见的症状和前，肺癌一定是一种常见的症状和前，肺癌一定是一种常见的症状和前，肺癌一定是一种常见的症状和前，肺癌一定是一种常见的�
40 125 2.054473638534546
prompt= 增龄性黄斑变性的病因是什么？(一)发病原因增龄性�
chosen= 
gen= ��斑变，，(一)发病原因增龄性黄斑变，，(一)发病原因增龄性黄斑变，，(一)发病原因增龄性黄斑变，，(一)发病原因增龄性黄斑变，，(一)发病原因增龄性黄斑变，，(一)发病原因增龄性黄斑变，，(一)发病原因增龄性黄斑变，，(一)发病原因�
50 125 1.4502208232879639
prompt= 口腔白斑治疗方法？临床上病理上不能诊断为其他疾�
chosen= 
gen= �理可以口腔白斑治疗方法，临床上病理上是一个人经病理可以口腔白斑治疗方法，临床上